# LeetCode 3977 — Minimum Time to Reach Target With Limited Power bằng Python

- Link: https://leetcode.com/problems/minimum-time-to-reach-target-with-limited-power/
- Độ khó: **Hard**
- Chủ đề: Đồ thị có hướng, đường đi ngắn nhất (Dijkstra), DP trạng thái `(đỉnh, năng lượng còn lại)`, hàng đợi ưu tiên (heap)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minTimeMaxPower(self, n: int, edges: list[list[int]], power: int, cost: list[int], source: int, target: int) -> list[int]:
```

## 1. Giới thiệu & mục tiêu học

Một tín hiệu xuất phát từ `source` ở thời điểm `0` với `power` đơn vị năng lượng. Mỗi lần đi qua một cạnh có hướng thì tốn thời gian `t`, và để **rời** khỏi đỉnh `u` bằng bất kỳ cạnh nào, tín hiệu phải trả `cost[u]` năng lượng (và phải còn **ít nhất** `cost[u]`). Ta cần: thời gian **ngắn nhất** để tới `target`, và trong các đường đạt thời gian ngắn nhất đó, năng lượng còn lại **lớn nhất**.

Đây là bài "đường đi ngắn nhất có thêm một tài nguyên bị giới hạn". Khi có thêm tài nguyên, **đỉnh** chưa đủ làm trạng thái: phải thêm "còn bao nhiêu năng lượng".

Sau notebook này bạn sẽ:

1. Nhận ra khi nào "trạng thái = đỉnh" không đủ và phải mở rộng thành `(đỉnh, năng lượng)`.
2. Nắm Dijkstra với `heapq`, mẹo dùng số âm để "ưu tiên năng lượng lớn hơn" khi thời gian bằng nhau.
3. Thấy một cách làm **không cần heap**: vì mỗi bước tốn ít nhất `1` năng lượng, năng lượng luôn giảm ngặt → xử lý theo thứ tự năng lượng giảm dần (DP trên DAG).
4. Viết 4 lời giải (vét cạn DFS, DP theo lớp năng lượng, Dijkstra `(đỉnh, năng lượng)`, Dijkstra cắt tỉa trội) và đối chiếu bằng test ngẫu nhiên.
5. Phân tích `O(power · (n + m))` và `O((n + m) · power · log(n · power))`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Đồ thị có hướng `n` đỉnh `0..n-1`; `edges[i] = [u, v, t]` là cạnh từ `u` tới `v` mất `t` giây. Mảng `cost[u]` là năng lượng cần để chuyển tiếp tín hiệu qua **một** cạnh đi ra từ `u` (dù chọn cạnh nào). Tín hiệu bắt đầu ở `source`, thời gian `0`, năng lượng `power`.

Quy tắc:

- Chỉ được đi qua cạnh xuất phát từ `u` nếu năng lượng hiện có `>= cost[u]`.
- Khi tới một đỉnh **không** tốn năng lượng; chỉ khi **rời** đỉnh `u` mới trừ `cost[u]`.
- Mỗi cạnh `[u, v, t]` cộng `t` vào tổng thời gian.

Trả về mảng `[thời gian nhỏ nhất đến target, năng lượng còn lại lớn nhất trong các đường đạt thời gian đó]`; nếu không tới được thì `[-1, -1]`.

**Ví dụ 1:** `n=5, edges=[[0,1,1],[1,4,1],[0,2,1],[2,3,1],[3,4,1]], power=4, cost=[2,3,1,1,1], source=0, target=4` → `[3, 0]`.
Đường `0→1→4` nhanh hơn (2 giây) nhưng sau khi rời 0 chỉ còn `4-2=2 < cost[1]=3` nên không đi tiếp được. Đường `0→2→3→4` tốn `3` giây và tổng năng lượng tiêu hao là `cost[0] + cost[2] + cost[3] = 2 + 1 + 1 = 4`, còn `0`.

**Ví dụ 2:** `n=3, edges=[[0,1,2],[1,2,2],[2,0,2]], power=3, cost=[1,1,1], source=1, target=1` → `[0, 3]` (đã ở đích).

**Ví dụ 3:** `n=4, edges=[[0,1,3],[2,3,4]], power=3, cost=[1,1,1,1], source=0, target=3` → `[-1, -1]`.

**Ràng buộc**

- `1 <= n <= 1000`, `0 <= edges.length <= 1000`
- `0 <= u, v <= n - 1`, `1 <= t <= 10^9`
- `1 <= power <= 1000`, `cost.length == n`, `1 <= cost[i] <= 2000`
- `0 <= source, target <= n - 1`

In [1]:
examples = [
    (dict(n=5, edges=[[0,1,1],[1,4,1],[0,2,1],[2,3,1],[3,4,1]], power=4, cost=[2,3,1,1,1], source=0, target=4), [3, 0]),
    (dict(n=3, edges=[[0,1,2],[1,2,2],[2,0,2]], power=3, cost=[1,1,1], source=1, target=1), [0, 3]),
    (dict(n=4, edges=[[0,1,3],[2,3,4]], power=3, cost=[1,1,1,1], source=0, target=3), [-1, -1]),
]
for args, expected in examples:
    print("n =", args["n"], "| power =", args["power"], "| source -> target =", args["source"], "->", args["target"], "| mong đợi:", expected)

n = 5 | power = 4 | source -> target = 0 -> 4 | mong đợi: [3, 0]
n = 3 | power = 3 | source -> target = 1 -> 1 | mong đợi: [0, 3]
n = 4 | power = 3 | source -> target = 0 -> 3 | mong đợi: [-1, -1]


## 3. Kiến thức nền

### 3.1. Danh sách kề (adjacency list)

Đồ thị có hướng lưu bằng danh sách kề: `g[u]` là danh sách các cặp `(v, t)` của cạnh đi ra từ `u`.

In [2]:
n = 4
edges = [[0, 1, 5], [0, 2, 2], [2, 1, 1], [1, 3, 4]]
g = [[] for _ in range(n)]          # n danh sách rỗng độc lập
for u, v, t in edges:               # giải nén từng cạnh thành u, v, t
    g[u].append((v, t))
print(g)                            # [[(1, 5), (2, 2)], [(3, 4)], [(1, 1)], []]
for v, t in g[0]:
    print("từ 0 đi tới", v, "mất", t, "giây")

[[(1, 5), (2, 2)], [(3, 4)], [(1, 1)], []]
từ 0 đi tới 1 mất 5 giây
từ 0 đi tới 2 mất 2 giây


### 3.2. Hàng đợi ưu tiên `heapq` và so sánh tuple

`heapq` trong Python là **min-heap**: `heappop` lấy phần tử **nhỏ nhất**. Khi phần tử là tuple, Python so sánh **từng phần** từ trái sang phải. Vì vậy `(thời gian, -năng lượng, đỉnh)` sẽ:

1. ưu tiên thời gian nhỏ hơn;
2. nếu bằng nhau, ưu tiên `-năng lượng` nhỏ hơn, tức **năng lượng lớn hơn**.

Đó là mẹo "đổi dấu để biến min-heap thành max-heap" cho thành phần thứ hai.

In [3]:
import heapq

pq = []
heapq.heappush(pq, (5, -1, "A"))
heapq.heappush(pq, (3, -2, "B"))
heapq.heappush(pq, (3, -7, "C"))      # cùng thời gian 3 nhưng năng lượng 7 > 2
heapq.heappush(pq, (4, -9, "D"))
while pq:
    d, negp, name = heapq.heappop(pq)
    print("lấy ra:", name, "thời gian", d, "năng lượng", -negp)
# thứ tự: C (3, 7) -> B (3, 2) -> D (4, 9) -> A (5, 1)

lấy ra: C thời gian 3 năng lượng 7
lấy ra: B thời gian 3 năng lượng 2
lấy ra: D thời gian 4 năng lượng 9
lấy ra: A thời gian 5 năng lượng 1


### 3.3. Vô cực `INF` và bảng 2 chiều `dist[u][p]`

Để nói "chưa đến được", ta khởi tạo khoảng cách bằng `float('inf')` (hoặc một số rất lớn). Với trạng thái có hai thành phần, `dist[u][p]` là khoảng cách nhỏ nhất tới đỉnh `u` khi còn đúng `p` năng lượng. Kích thước bảng: `n × (power + 1)`.

In [4]:
INF = float("inf")
n, power = 3, 4
dist = [[INF] * (power + 1) for _ in range(n)]     # KHÔNG dùng [[INF]*(power+1)]*n
dist[0][power] = 0
print(dist[0])                 # [inf, inf, inf, inf, 0]
print(dist[1][2] == INF)       # True: chưa đến được đỉnh 1 với 2 năng lượng
print(min(5, INF), 10**18 < INF)

[inf, inf, inf, inf, 0]
True
5 True


### 3.4. Dijkstra "lười" (lazy deletion)

Dijkstra chuẩn: luôn lấy trạng thái có khoảng cách nhỏ nhất ra khỏi heap, rồi cập nhật các hàng xóm. Ta cho phép một trạng thái nằm trong heap nhiều lần (mỗi lần cải thiện lại `dist` đẩy thêm một bản mới). Khi lấy ra mà `d > dist[...]` thì đó là bản cũ → **bỏ qua** (lazy deletion). Điều này an toàn vì bản tốt hơn đã được xử lý trước.

## 4. Ý tưởng nền tảng / trực giác

Nếu đề chỉ hỏi thời gian ngắn nhất thì đó là Dijkstra thuần trên đỉnh. Nhưng **năng lượng ảnh hưởng tới việc đi tiếp được hay không**: đường ngắn nhất theo thời gian đôi khi hết năng lượng giữa chừng (ví dụ 1).

**Tính tay ví dụ 1** (`power = 4`):

| Bước | Vị trí | Năng lượng | Thời gian | Ghi chú |
|---|---|---|---|---|
| bắt đầu | 0 | 4 | 0 | cần `cost[0] = 2` để rời, đủ |
| `0 → 1` | 1 | 2 | 1 | muốn rời 1 cần `cost[1] = 3`, thiếu → kẹt |
| `0 → 2` | 2 | 2 | 1 | rời 2 cần `1` → đủ |
| `2 → 3` | 3 | 1 | 2 | rời 3 cần `1` → đủ |
| `3 → 4` | 4 | 0 | 3 | tới đích |

Cùng một đỉnh có thể tới với **nhiều mức năng lượng khác nhau** (và thời gian khác nhau). Vì vậy ta phải theo dõi cả hai: `(đỉnh, năng lượng còn lại)`.

Ta kiểm chứng nhận xét "đường nhanh nhất chưa chắc đi được" bằng một Dijkstra bỏ qua năng lượng.

In [5]:
import heapq

def dijkstra_ignore_power(n, edges, source, target):
    g = [[] for _ in range(n)]
    for u, v, t in edges:
        g[u].append((v, t))
    dist = [float("inf")] * n
    dist[source] = 0
    pq = [(0, source)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, t in g[u]:
            if d + t < dist[v]:
                dist[v] = d + t
                heapq.heappush(pq, (d + t, v))
    return dist[target]

args = examples[0][0]
print("bỏ qua năng lượng:", dijkstra_ignore_power(args["n"], args["edges"], args["source"], args["target"]), "(nhưng đáp án thật là 3)")

bỏ qua năng lượng: 2 (nhưng đáp án thật là 3)


## 5. Cách 1 — Vét cạn: DFS liệt kê mọi đường đi hợp lệ

**Ý tưởng.** Đệ quy `go(u, d, p)`: đang ở đỉnh `u`, đã mất `d` giây, còn `p` năng lượng.

- Nếu `u == target`: ghi nhận ứng viên `(d, p)`.
- Nếu `p >= cost[u]`: với mọi cạnh `(v, t)` đi ra từ `u`, gọi `go(v, d + t, p - cost[u])`.

Cuối cùng chọn ứng viên có `d` nhỏ nhất, hòa thì `p` lớn nhất. Vì `cost >= 1`, năng lượng giảm ít nhất `1` mỗi bước nên đệ quy **luôn dừng** (độ sâu `<= power`).

In [6]:
def brute_force(n, edges, power, cost, source, target):
    g = [[] for _ in range(n)]
    for u, v, t in edges:
        g[u].append((v, t))
    best = [None]      # (thời gian, -năng lượng) nhỏ nhất

    def go(u, d, p):
        if u == target:
            cand = (d, -p)
            if best[0] is None or cand < best[0]:
                best[0] = cand
        if p >= cost[u]:
            for v, t in g[u]:
                go(v, d + t, p - cost[u])

    go(source, 0, power)
    return [-1, -1] if best[0] is None else [best[0][0], -best[0][1]]

for args, expected in examples:
    got = brute_force(**args)
    print(got, expected)
    assert got == expected

[3, 0] [3, 0]
[0, 3] [0, 3]
[-1, -1] [-1, -1]


**Độ phức tạp.** Số đường đi có thể là `deg^power` — mũ. Ví dụ đồ thị hai chiều dày với `power = 1000` là bất khả thi. Chỉ dùng để đối chiếu với input nhỏ.

**Nút thắt.** Nhiều đường đi khác nhau cùng đến một trạng thái `(đỉnh, năng lượng)` với thời gian khác nhau; ta chỉ cần giữ **thời gian nhỏ nhất** cho mỗi trạng thái. Đó là ý tưởng DP/đường đi ngắn nhất trên đồ thị trạng thái.

## 6. Bước đột phá

**Insight 1 — đồ thị trạng thái.** Mỗi trạng thái là `(u, p)`. Từ `(u, p)` với `p >= cost[u]`, mỗi cạnh `(u → v, t)` cho một cạnh sang `(v, p - cost[u])` có trọng số `t`. Bài toán thành: tìm đường ngắn nhất trong đồ thị trạng thái từ `(source, power)` tới **mọi** trạng thái `(target, *)`.

**Insight 2 — số trạng thái nhỏ.** Chỉ có `n × (power + 1) <= 1000 × 1001 ≈ 10^6` trạng thái và `m × (power + 1) <= 10^6` cạnh trạng thái: vừa với Python.

**Insight 3 — hai cách duyệt.**

- Trọng số `t >= 1` dương → **Dijkstra** hoạt động trên đồ thị trạng thái.
- Hơn nữa `cost >= 1` nên năng lượng **giảm ngặt** sau mỗi bước ⇒ đồ thị trạng thái **không có chu trình** (DAG), và thứ tự `p` giảm dần là một thứ tự tô-pô. Ta không cần heap!

**Insight 4 — hai tiêu chí.** Với đích `target`, ta muốn `(thời gian nhỏ nhất, năng lượng lớn nhất)`. Trong Dijkstra, dùng khóa `(d, -p)`; trong DP theo lớp năng lượng, duyệt `p` giảm dần và chỉ cập nhật khi thời gian **nhỏ hơn hẳn**.

**Quy tắc "nếu… thì…":**

- Nếu `p < cost[u]` thì trạng thái `(u, p)` là ngõ cụt (không chuyển tiếp được), nhưng vẫn có thể là đích nếu `u == target`.
- Nếu hai trạng thái cùng đỉnh, trạng thái có `d` nhỏ hơn **và** `p` lớn hơn (hoặc bằng) thì **trội hơn**: bỏ trạng thái kia cũng không mất đáp án (dùng ở cách 4).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP theo lớp năng lượng giảm dần (không cần heap)

**Giải thích từng đoạn.**

1. `dist[p][u]` = thời gian nhỏ nhất tới `u` khi còn đúng `p` năng lượng. Khởi tạo `dist[power][source] = 0`.
2. Duyệt `p` từ `power` xuống `0`. Khi tới lớp `p`, mọi trạng thái của lớp này đã nhận đủ giá trị từ các lớp năng lượng cao hơn (vì mỗi bước giảm năng lượng ít nhất `1`).
3. Với mỗi `u` có `dist[p][u]` hữu hạn và `p >= cost[u]`: relax mọi cạnh sang `dist[p - cost[u]][v]`.
4. Cuối cùng quét `p` từ cao xuống thấp trên cột `target`: chọn thời gian nhỏ nhất; vì duyệt `p` giảm dần và chỉ cập nhật khi thời gian **nhỏ hơn hẳn**, hòa sẽ giữ `p` lớn hơn.

In [7]:
INF = float("inf")

def layered_dp(n, edges, power, cost, source, target):
    g = [[] for _ in range(n)]
    for u, v, t in edges:
        g[u].append((v, t))
    dist = [[INF] * n for _ in range(power + 1)]       # dist[p][u]
    dist[power][source] = 0
    for p in range(power, -1, -1):                     # năng lượng giảm dần
        row = dist[p]
        for u in range(n):
            d = row[u]
            if d == INF or p < cost[u]:
                continue
            q = p - cost[u]
            nxt = dist[q]
            for v, t in g[u]:
                if d + t < nxt[v]:
                    nxt[v] = d + t
    best_t, best_p = INF, -1
    for p in range(power, -1, -1):
        if dist[p][target] < best_t:                   # chỉ cập nhật khi nhỏ hơn hẳn
            best_t, best_p = dist[p][target], p
    return [-1, -1] if best_t == INF else [best_t, best_p]

for args, expected in examples:
    got = layered_dp(**args)
    print(got, expected)
    assert got == expected

[3, 0] [3, 0]
[0, 3] [0, 3]
[-1, -1] [-1, -1]


**Độ phức tạp.** Với mỗi `p` ta quét mọi đỉnh và mọi cạnh một lần: `O(power · (n + m))` thời gian, `O(power · n)` bộ nhớ. Với `power = 1000`, `n = m = 1000`: khoảng `2 · 10^6` phép — rất nhanh.

### 7.2. Cách 3 — Dijkstra trên `(đỉnh, năng lượng)`

**Giải thích từng đoạn.**

1. `dist[u][p]` như trên (lưu theo `u` trước). Heap chứa `(d, -p, u)`.
2. Lấy ra `(d, -p, u)`. Nếu `u == target`: **trả về ngay** `[d, p]` — vì heap ưu tiên thời gian nhỏ rồi năng lượng lớn, đây chính là đáp án.
3. Nếu `d > dist[u][p]` (bản cũ) hoặc `p < cost[u]` (không đủ năng lượng để rời): bỏ qua.
4. Còn lại: `p2 = p - cost[u]`, với mỗi cạnh `(v, t)`, nếu `d + t < dist[v][p2]` thì cập nhật và đẩy `(d + t, -p2, v)`.

**Vì sao trả về ngay khi pop `target` đầu tiên là đúng?** Mọi trạng thái `(target, *)` có thời gian bằng `d_min` đều đã nằm trong heap trước khi bất kỳ trạng thái nào có thời gian `d_min` được lấy ra (vì mỗi cạnh có `t >= 1` nên chúng sinh ra từ các trạng thái có thời gian `< d_min`). Heap lấy ra theo `(d, -p)`, nên trạng thái đích đầu tiên có thời gian nhỏ nhất và, trong số đó, năng lượng lớn nhất.

In [8]:
def dijkstra_state(n, edges, power, cost, source, target):
    g = [[] for _ in range(n)]
    for u, v, t in edges:
        g[u].append((v, t))
    dist = [[INF] * (power + 1) for _ in range(n)]
    dist[source][power] = 0
    pq = [(0, -power, source)]
    while pq:
        d, negp, u = heapq.heappop(pq)
        p = -negp
        if u == target:
            return [d, p]
        if d > dist[u][p] or p < cost[u]:
            continue
        p2 = p - cost[u]
        for v, t in g[u]:
            nd = d + t
            if nd < dist[v][p2]:
                dist[v][p2] = nd
                heapq.heappush(pq, (nd, -p2, v))
    return [-1, -1]

for args, expected in examples:
    got = dijkstra_state(**args)
    print(got, expected)
    assert got == expected

[3, 0] [3, 0]
[0, 3] [0, 3]
[-1, -1] [-1, -1]


**Độ phức tạp.** Có tối đa `n · (power + 1)` trạng thái, `m · (power + 1)` lần đẩy vào heap, mỗi lần `O(log)` ⇒ `O((n + m) · power · log(n · power))` thời gian, `O(n · power)` bộ nhớ.

### 7.3. Cách 4 — Dijkstra với cắt tỉa trội (dominance)

**Ý tưởng.** Với mỗi đỉnh `u` chỉ cần nhớ `best[u]` = năng lượng **lớn nhất** đã gặp ở các trạng thái đã lấy ra khỏi heap. Vì heap lấy theo thời gian không giảm, nếu một trạng thái `(d, p, u)` mới được lấy ra có `p <= best[u]` thì tồn tại trạng thái trước đó `(d' <= d, p' >= p)` tại `u`: mọi đường đi tiếp từ trạng thái mới có thể thực hiện bởi trạng thái cũ với thời gian không lớn hơn và năng lượng không nhỏ hơn. Do đó bỏ được mà không mất đáp án.

**Vì sao vẫn đúng với tiêu chí `(thời gian, năng lượng)`?** Với cặp tiêu chí đó, trạng thái trội cho kết quả `(d_total', p_total')` thỏa `d_total' <= d_total` và `p_total' >= p_total` ở mọi đích khả dĩ; vậy không bao giờ tệ hơn.

In [9]:
def dijkstra_pruned(n, edges, power, cost, source, target):
    g = [[] for _ in range(n)]
    for u, v, t in edges:
        g[u].append((v, t))
    best = [-1] * n                      # năng lượng lớn nhất đã xử lý tại mỗi đỉnh
    pq = [(0, -power, source)]
    while pq:
        d, negp, u = heapq.heappop(pq)
        p = -negp
        if u == target:
            return [d, p]
        if p <= best[u]:                 # bị trội bởi trạng thái đã xử lý
            continue
        best[u] = p
        if p < cost[u]:
            continue
        p2 = p - cost[u]
        for v, t in g[u]:
            if p2 > best[v]:             # cắt tỉa sớm khi đẩy vào heap
                heapq.heappush(pq, (d + t, -p2, v))
    return [-1, -1]

for args, expected in examples:
    got = dijkstra_pruned(**args)
    print(got, expected)
    assert got == expected

[3, 0] [3, 0]
[0, 3] [0, 3]
[-1, -1] [-1, -1]


**Độ phức tạp.** Trường hợp xấu vẫn `O((n + m) · power · log)` (nhiều trạng thái không trội nhau), nhưng thực tế thường nhỏ hơn nhiều. Bộ nhớ `O(n + số phần tử trong heap)`.

## 8. TRACE từng bước

Ta trace `dijkstra_state` trên ví dụ 1. In mỗi lần `heappop`: trạng thái `(d, p, u)`, rồi hành động (bỏ qua / dừng / mở rộng) và các lần đẩy mới.

Bảng mong đợi (ví dụ 1, `cost = [2,3,1,1,1]`):

| Lần | Lấy ra `(d, p, u)` | Hành động |
|---|---|---|
| 1 | `(0, 4, 0)` | rời 0: `p2 = 2`; đẩy `(1,2,1)` và `(1,2,2)` |
| 2 | `(1, 2, 1)` | `p = 2 < cost[1] = 3` → ngõ cụt |
| 3 | `(1, 2, 2)` | rời 2: `p2 = 1`; đẩy `(2,1,3)` |
| 4 | `(2, 1, 3)` | rời 3: `p2 = 0`; đẩy `(3,0,4)` |
| 5 | `(3, 0, 4)` | `u == target` → trả `[3, 0]` |

In [10]:
def trace_dijkstra(n, edges, power, cost, source, target):
    g = [[] for _ in range(n)]
    for u, v, t in edges:
        g[u].append((v, t))
    dist = [[INF] * (power + 1) for _ in range(n)]
    dist[source][power] = 0
    pq = [(0, -power, source)]
    step = 0
    while pq:
        d, negp, u = heapq.heappop(pq)
        p = -negp
        step += 1
        head = f"#{step}: lấy ra (d={d}, p={p}, u={u})"
        if u == target:
            print(head, "-> u là đích, DỪNG. Đáp án", [d, p])
            return [d, p]
        if d > dist[u][p]:
            print(head, "-> bản cũ, bỏ qua"); continue
        if p < cost[u]:
            print(head, f"-> p={p} < cost[{u}]={cost[u]}: không rời được (ngõ cụt)"); continue
        p2 = p - cost[u]
        pushed = []
        for v, t in g[u]:
            nd = d + t
            if nd < dist[v][p2]:
                dist[v][p2] = nd
                heapq.heappush(pq, (nd, -p2, v))
                pushed.append((nd, p2, v))
        print(head, f"-> rời {u}, p2={p2}, đẩy {pushed}")
    print("heap rỗng -> [-1, -1]")
    return [-1, -1]

trace_dijkstra(**examples[0][0])

#1: lấy ra (d=0, p=4, u=0) -> rời 0, p2=2, đẩy [(1, 2, 1), (1, 2, 2)]
#2: lấy ra (d=1, p=2, u=1) -> p=2 < cost[1]=3: không rời được (ngõ cụt)
#3: lấy ra (d=1, p=2, u=2) -> rời 2, p2=1, đẩy [(2, 1, 3)]
#4: lấy ra (d=2, p=1, u=3) -> rời 3, p2=0, đẩy [(3, 0, 4)]
#5: lấy ra (d=3, p=0, u=4) -> u là đích, DỪNG. Đáp án [3, 0]


[3, 0]

In [11]:
# Trace thêm ví dụ 3 (không tới được) và một đồ thị có hai đường cùng thời gian nhưng năng lượng khác nhau
trace_dijkstra(**examples[2][0])
print("---")
tie = dict(n=4, edges=[[0,1,2],[1,3,2],[0,2,2],[2,3,2]], power=10, cost=[1,5,1,1], source=0, target=3)
trace_dijkstra(**tie)
print("brute_force:", brute_force(**tie))

#1: lấy ra (d=0, p=3, u=0) -> rời 0, p2=2, đẩy [(3, 2, 1)]
#2: lấy ra (d=3, p=2, u=1) -> rời 1, p2=1, đẩy []
heap rỗng -> [-1, -1]
---
#1: lấy ra (d=0, p=10, u=0) -> rời 0, p2=9, đẩy [(2, 9, 1), (2, 9, 2)]
#2: lấy ra (d=2, p=9, u=1) -> rời 1, p2=4, đẩy [(4, 4, 3)]
#3: lấy ra (d=2, p=9, u=2) -> rời 2, p2=8, đẩy [(4, 8, 3)]
#4: lấy ra (d=4, p=8, u=3) -> u là đích, DỪNG. Đáp án [4, 8]
brute_force: [4, 8]


Ở đồ thị "hòa thời gian" phía trên, hai đường `0→1→3` và `0→2→3` đều mất `4` giây, nhưng đường qua đỉnh `1` trừ `cost[1] = 5` còn đường qua đỉnh `2` chỉ trừ `1`. Nhờ khóa `(d, -p)`, trạng thái năng lượng lớn hơn được lấy ra trước nên Dijkstra trả đúng năng lượng còn lại lớn nhất là `8`.

## 9. Code nộp LeetCode

Dùng Dijkstra trên `(đỉnh, năng lượng)` (cách 3) có thêm cắt tỉa trạng thái bản cũ. Nếu bạn thích không cần heap thì cách 2 cũng nộp được, thậm chí gọn hơn. Ở đây nộp cách 3 để khớp với lời giải tham khảo.

In [12]:
import heapq
from math import inf

class Solution:
    def minTimeMaxPower(
        self,
        n: int,
        edges: list[list[int]],
        power: int,
        cost: list[int],
        source: int,
        target: int,
    ) -> list[int]:
        g = [[] for _ in range(n)]
        for u, v, t in edges:
            g[u].append((v, t))
        dist = [[inf] * (power + 1) for _ in range(n)]
        dist[source][power] = 0
        pq = [(0, -power, source)]            # (thời gian, -năng lượng, đỉnh)
        while pq:
            d, p, u = heapq.heappop(pq)
            p = -p
            if u == target:
                return [d, p]
            if d > dist[u][p] or p < cost[u]:
                continue
            p -= cost[u]
            for v, t in g[u]:
                nd = d + t
                if nd < dist[v][p]:
                    dist[v][p] = nd
                    heapq.heappush(pq, (nd, -p, v))
        return [-1, -1]


sol = Solution()
for args, expected in examples:
    got = sol.minTimeMaxPower(**args)
    print(got, expected)
    assert got == expected

[3, 0] [3, 0]
[0, 3] [0, 3]
[-1, -1] [-1, -1]


## 10. Vì sao thuật toán đúng

**Đồ thị trạng thái mô phỏng đúng luật.** Một đường đi hợp lệ của tín hiệu từ `source` tương ứng 1-1 với một đường đi trong đồ thị trạng thái bắt đầu ở `(source, power)`: cạnh `(u,p) → (v,p-cost[u])` tồn tại khi và chỉ khi có cạnh `u→v` và `p >= cost[u]`. Trọng số là thời gian `t`.

**Dijkstra đúng.** Trọng số dương nên Dijkstra tìm đúng khoảng cách nhỏ nhất `dist` tới mọi trạng thái: khi một trạng thái được lấy ra với `d == dist[u][p]` thì không còn đường nào ngắn hơn (mọi đường khác phải đi qua trạng thái còn trong heap với khoảng cách `>= d`).

**Tiêu chí thứ hai.** Hai trạng thái cùng đỉnh khác `p` là hai nút khác nhau trong đồ thị trạng thái nên không "đè" nhau. Với đích: thời gian nhỏ nhất là `min_p dist[target][p]`, và đáp án thứ hai là `max` của `p` thỏa `dist[target][p] = thời gian nhỏ nhất`. Heap sắp theo `(d, -p)` nên trạng thái đích đầu tiên bị lấy ra chính là cặp này (lập luận ở 7.2).

**DP theo lớp (cách 2) đúng** vì cạnh trạng thái luôn đi từ `p` sang `p - cost[u] < p`: duyệt `p` giảm dần là thứ tự tô-pô nên khi xét lớp `p`, `dist[p][*]` đã hoàn tất.

**Cắt tỉa trội (cách 4) đúng** vì quan hệ "trội" (thời gian ≤ và năng lượng ≥) bảo toàn khả năng đi tiếp: từ năng lượng lớn hơn, mọi cạnh có thể đi cũng đi được, và còn lại nhiều năng lượng hơn.

**Trường hợp `source == target`** được xử lý bởi lần pop đầu tiên: trả `[0, power]` mà không cần đủ `cost[source]`, đúng như ví dụ 2.

## 11. Phân tích Time/Space

Đặt `n` = số đỉnh, `m` = số cạnh, `P = power`.

| Cách | Time | Space |
|---|---|---|
| Vét cạn DFS | mũ theo `P` | `O(P)` |
| DP theo lớp năng lượng | `O(P · (n + m))` | `O(P · n)` |
| Dijkstra `(đỉnh, p)` | `O((n + m) · P · log(n · P))` | `O(n · P)` |
| Dijkstra cắt tỉa trội | `O((n + m) · P · log)` trường hợp xấu, thường nhỏ hơn | `O(n + heap)` |

Giới hạn `n, m, P <= 1000` ⇒ khoảng `10^6` trạng thái và `10^6` cạnh trạng thái. Với Dijkstra, mỗi lần đẩy heap tốn vài µs ⇒ vài giây trong Python ở test xấu nhất; cách 2 (không heap) thường nhanh hơn rõ rệt.

In [13]:
import time, random
rng = random.Random(1)
n = 500; P = 1000
# chuỗi 0->1->...->499 cộng thêm cạnh nhảy cách 2: m = 499 + 498 = 997 <= 1000, nhiều mức năng lượng khác nhau tại mỗi đỉnh
edges = [[i, i + 1, rng.randint(1, 10**9)] for i in range(n - 1)] + [[i, i + 2, rng.randint(1, 10**9)] for i in range(n - 2)]
cost = [rng.randint(1, 3) for _ in range(n)]
args = dict(n=n, edges=edges, power=P, cost=cost, source=0, target=n - 1)
print("số cạnh:", len(edges))
for name, fn in [("layered_dp", layered_dp), ("Solution (Dijkstra)", Solution().minTimeMaxPower), ("dijkstra_pruned", dijkstra_pruned)]:
    t0 = time.time()
    res = fn(**args)
    print(f"{name:22} -> {res}  ({time.time()-t0:.2f}s)")

số cạnh: 997
layered_dp             -> [95261325238, 377]  (0.09s)


Solution (Dijkstra)    -> [95261325238, 377]  (0.13s)
dijkstra_pruned        -> [95261325238, 377]  (0.03s)


## 12. Bộ test

Chiến lược test:

1. 3 ví dụ của đề (assert).
2. Biên: `source == target`; không có cạnh; năng lượng đúng bằng tổng chi phí cần; năng lượng thiếu `1`; cạnh song song; vòng lặp (chu trình) và tự trỏ.
3. Phản ví dụ: đường ngắn nhất theo thời gian nhưng hết năng lượng.
4. Đối chiếu `brute_force`, `layered_dp`, `dijkstra_state`, `dijkstra_pruned`, `Solution` trên nhiều đồ thị ngẫu nhiên nhỏ (seed cố định) — đáp án duy nhất nên so sánh bằng `==`.

In [14]:
sol = Solution()
for args, expected in examples:
    assert sol.minTimeMaxPower(**args) == expected

# biên
assert sol.minTimeMaxPower(n=1, edges=[], power=5, cost=[3], source=0, target=0) == [0, 5]
assert sol.minTimeMaxPower(n=2, edges=[], power=5, cost=[1, 1], source=0, target=1) == [-1, -1]
# năng lượng đúng bằng chi phí rời đỉnh 0: tới đích còn 0
assert sol.minTimeMaxPower(n=2, edges=[[0, 1, 7]], power=2, cost=[2, 9], source=0, target=1) == [7, 0]
# thiếu 1 năng lượng
assert sol.minTimeMaxPower(n=2, edges=[[0, 1, 7]], power=1, cost=[2, 9], source=0, target=1) == [-1, -1]
# cạnh song song: chọn cạnh nhanh hơn
assert sol.minTimeMaxPower(n=2, edges=[[0, 1, 9], [0, 1, 4]], power=3, cost=[1, 1], source=0, target=1) == [4, 2]
# tự trỏ và chu trình không giúp ích
assert sol.minTimeMaxPower(n=3, edges=[[0, 0, 1], [0, 1, 5], [1, 0, 1], [1, 2, 3]], power=4, cost=[1, 1, 1], source=0, target=2) == [8, 2]
print("test cố định OK")

test cố định OK


In [15]:
import random
rng = random.Random(20240601)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 6)
    m = rng.randint(0, 10)
    edges = [[rng.randrange(n), rng.randrange(n), rng.randint(1, 5)] for _ in range(m)]
    power = rng.randint(1, 8)
    cost = [rng.randint(1, 4) for _ in range(n)]
    source, target = rng.randrange(n), rng.randrange(n)
    args = dict(n=n, edges=edges, power=power, cost=cost, source=source, target=target)
    expected = brute_force(**args)
    assert layered_dp(**args) == expected, args
    assert dijkstra_state(**args) == expected, args
    assert dijkstra_pruned(**args) == expected, args
    assert Solution().minTimeMaxPower(**args) == expected, args
    cnt += 1
print("đã đối chiếu", cnt, "đồ thị ngẫu nhiên nhỏ: tất cả khớp vét cạn")

đã đối chiếu 1500 đồ thị ngẫu nhiên nhỏ: tất cả khớp vét cạn


In [16]:
# Đồ thị lớn hơn (vét cạn không chạy nổi): đối chiếu 4 cách DP/Dijkstra với nhau
rng = random.Random(777)
for _ in range(60):
    n = rng.randint(10, 60)
    m = rng.randint(n, 3 * n)
    edges = [[rng.randrange(n), rng.randrange(n), rng.randint(1, 10**9)] for _ in range(m)]
    power = rng.randint(5, 60)
    cost = [rng.randint(1, 6) for _ in range(n)]
    s, t = rng.randrange(n), rng.randrange(n)
    args = dict(n=n, edges=edges, power=power, cost=cost, source=s, target=t)
    a = layered_dp(**args)
    assert a == dijkstra_state(**args) == dijkstra_pruned(**args) == Solution().minTimeMaxPower(**args), args
print("4 cách khớp nhau trên 60 đồ thị vừa (trọng số lớn tới 10^9)")

4 cách khớp nhau trên 60 đồ thị vừa (trọng số lớn tới 10^9)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dijkstra chỉ theo đỉnh, bỏ qua năng lượng

**Code sai:** `dist[u]` chỉ là thời gian; coi đỉnh đã "xong" khi lấy ra lần đầu.
**Vì sao sai:** đường nhanh nhất tới `u` có thể còn ít năng lượng (vì đi qua đỉnh tốn nhiều năng lượng), và sau đó không đi tiếp được; trong khi đường chậm hơn nhưng rẻ năng lượng lại đi tiếp được. Đồ thị `trap` ở cell dưới minh họa đúng tình huống đó.
**Code đúng:** trạng thái là `(u, p)`.

In [17]:
def wrong_node_only(n, edges, power, cost, source, target):
    g = [[] for _ in range(n)]
    for u, v, t in edges:
        g[u].append((v, t))
    done = [False] * n
    pq = [(0, -power, source)]
    while pq:
        d, negp, u = heapq.heappop(pq)
        p = -negp
        if done[u]:
            continue
        done[u] = True                      # SAI: bỏ qua các mức năng lượng khác
        if u == target:
            return [d, p]
        if p < cost[u]:
            continue
        for v, t in g[u]:
            heapq.heappush(pq, (d + t, -(p - cost[u]), v))
    return [-1, -1]

# đồ thị bẫy: đường 0->2->1 nhanh (2 giây) nhưng đi qua đỉnh 2 tốn 5 năng lượng, tới đỉnh 1 chỉ còn 1 < cost[1] = 3;
# đường 0->3->4->1 chậm hơn (3 giây) nhưng rẻ năng lượng, tới đỉnh 1 còn 4 năng lượng -> đi tiếp được
trap = dict(n=6, edges=[[0,2,1],[2,1,1],[0,3,1],[3,4,1],[4,1,1],[1,5,1]], power=7, cost=[1,3,5,1,1,1], source=0, target=5)
print("sai :", wrong_node_only(**trap), " đúng:", Solution().minTimeMaxPower(**trap), " vét cạn:", brute_force(**trap))

sai : [-1, -1]  đúng: [4, 1]  vét cạn: [4, 1]


### Lỗi 2 — Trừ năng lượng khi *đến* đỉnh thay vì khi *rời*

**Code sai:** `p2 = p - cost[v]`.
**Vì sao sai:** đề nói "no power is consumed when the signal arrives at a node"; chi phí là của đỉnh **gửi đi** `cost[u]`. Hệ quả: đích không bao giờ bị trừ.
**Code đúng:** `p2 = p - cost[u]`, và kiểm tra `p >= cost[u]` trước khi rời `u`.

In [18]:
def wrong_charge_on_arrival(n, edges, power, cost, source, target):
    g = [[] for _ in range(n)]
    for u, v, t in edges:
        g[u].append((v, t))
    dist = [[INF] * (power + 1) for _ in range(n)]
    dist[source][power] = 0
    pq = [(0, -power, source)]
    while pq:
        d, negp, u = heapq.heappop(pq)
        p = -negp
        if u == target:
            return [d, p]
        if d > dist[u][p]:
            continue
        for v, t in g[u]:
            p2 = p - cost[v]                    # SAI: trừ theo đỉnh đến
            if p2 >= 0 and d + t < dist[v][p2]:
                dist[v][p2] = d + t
                heapq.heappush(pq, (d + t, -p2, v))
    return [-1, -1]

print("sai :", wrong_charge_on_arrival(**examples[0][0]), " đúng:", Solution().minTimeMaxPower(**examples[0][0]))

sai : [2, 0]  đúng: [3, 0]


### Lỗi 3 — Quên đẩy `-p` (hoặc đẩy `p`) vào heap nên không ưu tiên năng lượng lớn khi hòa thời gian

**Code sai:** `heappush(pq, (nd, p2, v))`.
**Vì sao sai:** khi thời gian bằng nhau, heap lấy `p` **nhỏ** trước; trả về ngay đích đầu tiên có thể là mức năng lượng thấp.
**Code đúng:** đẩy `-p2`.

In [19]:
def wrong_min_power_first(n, edges, power, cost, source, target):
    g = [[] for _ in range(n)]
    for u, v, t in edges:
        g[u].append((v, t))
    dist = [[INF] * (power + 1) for _ in range(n)]
    dist[source][power] = 0
    pq = [(0, power, source)]                   # SAI: không đổi dấu
    while pq:
        d, p, u = heapq.heappop(pq)
        if u == target:
            return [d, p]
        if d > dist[u][p] or p < cost[u]:
            continue
        p2 = p - cost[u]
        for v, t in g[u]:
            if d + t < dist[v][p2]:
                dist[v][p2] = d + t
                heapq.heappush(pq, (d + t, p2, v))
    return [-1, -1]

print("sai :", wrong_min_power_first(**tie), " đúng:", Solution().minTimeMaxPower(**tie))

sai : [4, 4]  đúng: [4, 8]


### Lỗi 4 — Trả về khi *đẩy* đích vào heap, không phải khi *lấy ra*

**Code sai:** thấy `v == target` lúc relax thì trả luôn.
**Vì sao sai:** lúc đẩy chưa chắc đã tối ưu (có thể có đường đến đích nhanh hơn hoặc nhiều năng lượng hơn chưa được xét).
**Code đúng:** trả về khi `heappop` ra trạng thái đích.

## 14. Mẹo & nhận dạng pattern

- **"Đường ngắn nhất + tài nguyên có hạn (xăng, năng lượng, số lần đổi...)"** → mở rộng trạng thái thành `(đỉnh, tài nguyên còn lại)`.
- Nếu tài nguyên **giảm ngặt** mỗi bước và miền nhỏ (`<= 1000`) → đồ thị trạng thái là DAG, có thể DP theo thứ tự tài nguyên, không cần heap.
- **Hai tiêu chí lexicographic** ("nhỏ nhất rồi lớn nhất") → khóa heap `(a, -b)`.
- **Trội (dominance):** cùng đỉnh, ít thời gian hơn & nhiều năng lượng hơn thì loại trạng thái còn lại — rất hay trong bài có 2 chiều.
- Luôn nghĩ "trả về khi lấy ra khỏi heap", không phải lúc đẩy vào.
- Ghi nhớ giới hạn: `n·power ≈ 10^6` là số trạng thái tối đa — vẫn ổn trong Python nếu mỗi trạng thái `O(1)` thao tác thật sự.

## 15. Biến thể

1. **Chi phí tính theo từng cạnh** (`cost` nằm trên cạnh): cạnh trạng thái `(u,p) → (v,p - c)`; vẫn Dijkstra/DP theo lớp (nếu `c >= 1`).
2. **Chi phí có thể bằng 0:** đồ thị trạng thái có thể có chu trình trong cùng một lớp `p` → bắt buộc Dijkstra (hoặc Dijkstra trong từng lớp).
3. **Năng lượng có thể hồi phục tại một số đỉnh:** `p` tăng (cap ở `power`), không còn DAG → Dijkstra trên `(u, p)`.
4. **Tối thiểu năng lượng tiêu hao trong các đường ngắn nhất:** thực chất đã là bài này (`power - answer[1]`).
5. **Giới hạn số cạnh đi (k bước) thay cho năng lượng:** Bellman-Ford theo lớp `k`.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn DFS | liệt kê mọi đường đi | mũ | `O(P)` | kiểm chứng input rất nhỏ |
| 2. DP theo lớp năng lượng | DAG theo `p` giảm dần | `O(P·(n+m))` | `O(P·n)` | khi `cost >= 1`: nhanh & đơn giản nhất |
| 3. Dijkstra `(u, p)` | đường ngắn nhất trên đồ thị trạng thái | `O((n+m)·P·log)` | `O(n·P)` | tổng quát (kể cả `cost = 0`, hồi phục) |
| 4. Dijkstra cắt tỉa trội | bỏ trạng thái bị trội | thường nhanh hơn 3 | `O(n + heap)` | khi muốn giảm trạng thái |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** Chạy lần lượt `heappush(pq, (2, -3, 'x'))`, `(2, -5, 'y')`, `(1, -1, 'z')` rồi `heappop` ba lần. Thứ tự tên là gì?

**Bài 2 (tính tay).** `n=3, edges=[[0,1,4],[1,2,1],[0,2,10]], power=3, cost=[1,2,1], source=0, target=2`. Đáp án là gì?

**Bài 3 (điền TODO).** Hoàn thiện `my_layered(n, edges, power, cost, source, target)` theo cách 2.

**Bài 4 (tự viết lại).** Viết `my_dijkstra(...)` theo cách 3 mà không nhìn code nộp.

**Bài 5 (biến thể).** Viết `max_power_any_time(n, edges, power, cost, source, target)`: **bỏ qua thời gian**, trả năng lượng còn lại **lớn nhất** có thể khi tới `target` (hoặc `-1` nếu không tới được).

In [20]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_layered(n, edges, power, cost, source, target):
    # TODO 1: dựng danh sách kề g
    # TODO 2: dist[p][u] = INF, dist[power][source] = 0
    # TODO 3: for p từ power xuống 0: for u: nếu dist hữu hạn và p >= cost[u] thì relax sang dist[p - cost[u]][v]
    # TODO 4: quét p từ cao xuống thấp ở cột target, chỉ cập nhật khi thời gian nhỏ hơn hẳn
    pass

# Bài 4 — TODO
def my_dijkstra(n, edges, power, cost, source, target):
    # TODO: heap (d, -p, u); trả về khi pop đỉnh target
    pass

# Bài 5 — TODO
def max_power_any_time(n, edges, power, cost, source, target):
    # TODO: reach[p][u] boolean theo p giảm dần; chọn p lớn nhất có reach[p][target]
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: heap lấy phần tử nhỏ nhất theo tuple.
- Mức 2: so sánh phần tử đầu (`2, 2, 1`) trước.
- Mức 3: `(1,-1,'z')` đầu tiên, rồi `(2,-5,'y')` (vì `-5 < -3`), rồi `(2,-3,'x')`: `z, y, x`.

**Bài 2.**
- Mức 1: liệt kê hai đường tới 2: `0→2` và `0→1→2`.
- Mức 2: `0→2`: rời 0 tốn 1 (còn 2), tới 2, thời gian 10. `0→1→2`: rời 0 còn 2; rời 1 cần `cost[1] = 2`, đủ (còn 0), tới 2; thời gian `4 + 1 = 5`.
- Mức 3: thời gian nhỏ nhất `5`, năng lượng còn `0` → `[5, 0]`.

**Bài 3.**
- Mức 1: `dist = [[INF]*n for _ in range(power+1)]`.
- Mức 2: vòng ngoài `for p in range(power, -1, -1)`.
- Mức 3: `q = p - cost[u]` và `if d + t < dist[q][v]: dist[q][v] = d + t`.

**Bài 4.**
- Mức 1: `dist[u][p]` và heap `(0, -power, source)`.
- Mức 2: bỏ qua nếu `d > dist[u][p]` hoặc `p < cost[u]`.
- Mức 3: đẩy `(nd, -p2, v)` khi cải thiện `dist[v][p2]`.

**Bài 5.**
- Mức 1: không cần thời gian → chỉ cần "có đến được không".
- Mức 2: `reach[power][source] = True`; từ `reach[p][u]` và `p >= cost[u]` bật `reach[p - cost[u]][v]`.
- Mức 3: quét `p` từ cao xuống thấp, gặp `reach[p][target]` đầu tiên thì trả `p`.

## 19. Đáp án tham khảo

In [21]:
# Bài 1
pq = []
for item in [(2, -3, "x"), (2, -5, "y"), (1, -1, "z")]:
    heapq.heappush(pq, item)
assert [heapq.heappop(pq)[2] for _ in range(3)] == ["z", "y", "x"]
# Bài 2
assert Solution().minTimeMaxPower(n=3, edges=[[0,1,4],[1,2,1],[0,2,10]], power=3, cost=[1,2,1], source=0, target=2) == [5, 0]

# Bài 3
def my_layered_ans(n, edges, power, cost, source, target):
    g = [[] for _ in range(n)]
    for u, v, t in edges:
        g[u].append((v, t))
    dist = [[INF] * n for _ in range(power + 1)]
    dist[power][source] = 0
    for p in range(power, -1, -1):
        for u in range(n):
            d = dist[p][u]
            if d == INF or p < cost[u]:
                continue
            q = p - cost[u]
            for v, t in g[u]:
                if d + t < dist[q][v]:
                    dist[q][v] = d + t
    bt, bp = INF, -1
    for p in range(power, -1, -1):
        if dist[p][target] < bt:
            bt, bp = dist[p][target], p
    return [-1, -1] if bt == INF else [bt, bp]

for args, expected in examples:
    assert my_layered_ans(**args) == expected
print("Bài 1–3 OK")

Bài 1–3 OK


In [22]:
# Bài 4
def my_dijkstra_ans(n, edges, power, cost, source, target):
    g = [[] for _ in range(n)]
    for u, v, t in edges:
        g[u].append((v, t))
    dist = [[INF] * (power + 1) for _ in range(n)]
    dist[source][power] = 0
    pq = [(0, -power, source)]
    while pq:
        d, negp, u = heapq.heappop(pq)
        p = -negp
        if u == target:
            return [d, p]
        if d > dist[u][p] or p < cost[u]:
            continue
        p2 = p - cost[u]
        for v, t in g[u]:
            if d + t < dist[v][p2]:
                dist[v][p2] = d + t
                heapq.heappush(pq, (d + t, -p2, v))
    return [-1, -1]

# Bài 5: bỏ qua thời gian, tối đa năng lượng còn lại
def max_power_any_time_ans(n, edges, power, cost, source, target):
    g = [[] for _ in range(n)]
    for u, v, t in edges:
        g[u].append(v)
    reach = [[False] * n for _ in range(power + 1)]
    reach[power][source] = True
    for p in range(power, -1, -1):
        for u in range(n):
            if reach[p][u] and p >= cost[u]:
                for v in g[u]:
                    reach[p - cost[u]][v] = True
    for p in range(power, -1, -1):
        if reach[p][target]:
            return p
    return -1

def brute_max_power(n, edges, power, cost, source, target):
    g = [[] for _ in range(n)]
    for u, v, t in edges:
        g[u].append(v)
    best = [-1]
    def go(u, p):
        if u == target:
            best[0] = max(best[0], p)
        if p >= cost[u]:
            for v in g[u]:
                go(v, p - cost[u])
    go(source, power)
    return best[0]

rng = random.Random(11)
for _ in range(300):
    n = rng.randint(1, 6); m = rng.randint(0, 9)
    edges = [[rng.randrange(n), rng.randrange(n), rng.randint(1, 5)] for _ in range(m)]
    power = rng.randint(1, 7); cost = [rng.randint(1, 3) for _ in range(n)]
    s, t = rng.randrange(n), rng.randrange(n)
    args = dict(n=n, edges=edges, power=power, cost=cost, source=s, target=t)
    assert my_dijkstra_ans(**args) == Solution().minTimeMaxPower(**args) == my_layered_ans(**args)
    assert max_power_any_time_ans(**args) == brute_max_power(**args)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template Dijkstra trên `(đỉnh, tài nguyên)` với hai tiêu chí:**

```python
g = [[] for _ in range(n)]
for u, v, t in edges:
    g[u].append((v, t))
dist = [[inf] * (power + 1) for _ in range(n)]
dist[source][power] = 0
pq = [(0, -power, source)]                 # (thời gian, -năng lượng, đỉnh)
while pq:
    d, p, u = heappop(pq); p = -p
    if u == target:                        # lấy ra mới trả, không phải lúc đẩy
        return [d, p]
    if d > dist[u][p] or p < cost[u]:      # bản cũ hoặc không đủ năng lượng để rời
        continue
    p -= cost[u]                           # trừ khi RỜI đỉnh u
    for v, t in g[u]:
        if d + t < dist[v][p]:
            dist[v][p] = d + t
            heappush(pq, (d + t, -p, v))
return [-1, -1]
```

**Checklist trước khi submit:**

- [ ] Trạng thái là `(đỉnh, năng lượng)`, không chỉ đỉnh.
- [ ] Trừ `cost[u]` khi **rời** `u`, kiểm tra `p >= cost[u]` trước.
- [ ] Đẩy `-p` để năng lượng lớn được ưu tiên khi hòa thời gian.
- [ ] Trả về khi **pop** đích, và trả `[-1, -1]` khi heap rỗng.
- [ ] `source == target` → `[0, power]` (xử lý bằng lần pop đầu).
- [ ] Không dùng `[[inf]*k]*n` khi tạo bảng 2 chiều.
- [ ] Nhớ rằng nếu `cost >= 1` thì còn có cách DP theo lớp năng lượng, không cần heap.
- [ ] Đã đối chiếu vét cạn trên đồ thị nhỏ ngẫu nhiên.